# Apply babyseg to the OpenNeuro open dataset

Great! Now you have successfully downloaded the dataset and installed BabySeg

Now we want to perform a segmentation.

This can be run on some computers CPU, depending on how much memory is available - but lets give it a go!

## Step 1: Set up the paths and where to save

Following BIDS format, we want to use the derivatives folder for our segmentations



In [ ]:

import os
import subprocess
from pathlib import Path
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import sys

babyseg_home = (Path("..") / "babyseg").resolve() # where babyseg is stored

openneuro_root = (Path("..") / "open_data" / "OpenNeuro").resolve() # where our data is saved

sub = "sub-01" # our subject ID

mri_path =  f"{sub}/anat/sub-01_T1w.nii.gz" # MRI path from the openneuro_root

derivatives_outpath = (openneuro_root / "derivatives" / sub / "anat" ).resolve() # Where we want to save

os.makedirs(derivatives_outpath,exist_ok=True)

babyseg_path = (Path(derivatives_outpath) / "babyseg_out.nii.gz" ).resolve() 


## Step 2: Run the babyseg segmentation on our OpenNeuro data

### Step 2a: for MAC or LINUX Machines:

The cells execute the program through python, but this can be done in bash too

```sh
export BABYSEG_MNT="../open_data/OpenNeuro";
export BABYSEG_TAG=0.0-cu126
export BABYSEG_SIF="$BABYSEG_HOME"

./babyseg -o "{babyseg_path}"  "{mri_path}" 
```

In [ ]:

# Set up environemnt paths
env = os.environ.copy()
env["BABYSEG_MNT"] = str(openneuro_root)
env["BABYSEG_TAG"] = env.get("BABYSEG_TAG", "0.0") # Default to CPU container for broader compatibility. Override by setting BABYSEG_TAG in env.
env["BABYSEG_SIF"] = str(babyseg_home)



# Use a single thread to reduce memory pressure on Docker Desktop defaults.
cmd = [sys.executable, "./babyseg", "-o", str(babyseg_path), str(mri_path)]
print("Running:", " ".join(cmd))
print("Using BABYSEG_TAG:", env["BABYSEG_TAG"])

# Run the segmentation

result = subprocess.run(
    cmd,
    env=env,
    text=True,
    capture_output=True,
    check=False,
 )


# Check running and print errors:
if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    if result.stderr:
        print(result.stderr)
    if result.returncode == 137:
        raise RuntimeError(
            "babyseg process was killed (exit 137). Increase Docker memory, keep -j 1, or close other Docker workloads."
        )
    raise RuntimeError(f"babyseg run failed with exit code {result.returncode}")


### Step 2b: for windows machines:

We are going to call our executable wrapper script here

In [ ]:

# Set up environemnt paths
env = os.environ.copy()
env["BABYSEG_MNT"] = str(openneuro_root)
env["BABYSEG_TAG"] = env.get("BABYSEG_TAG", "0.0") # Default to CPU container for broader compatibility. Override by setting BABYSEG_TAG in env.
env["BABYSEG_SIF"] = str(babyseg_home)


# Find the wrapper
wrapper = babyseg_home / "wrapper_windows.py"


# Use a single thread to reduce memory pressure on Docker Desktop defaults.
cmd = [sys.executable, str(wrapper), "-o", str(babyseg_path), str(mri_path)]
print("Running:", " ".join(cmd))
print("Using BABYSEG_TAG:", env["BABYSEG_TAG"])

# Run the segmentation
result = subprocess.run(
    cmd,
    env=env,
    text=True,
    capture_output=True,
    check=False,
 )


# Check running and print errors:
if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    if result.stderr:
        print(result.stderr)
    if result.returncode == 137:
        raise RuntimeError(
            "babyseg process was killed (exit 137). Increase Docker memory, keep -j 1, or close other Docker workloads."
        )
    raise RuntimeError(f"babyseg run failed with exit code {result.returncode}")


## Step 3: Look at the segmentation

In [ ]:
# Load in image and segmentation
image = nib.load(os.path.join( str(openneuro_root),mri_path)).get_fdata()
seg = nib.load(str(babyseg_path)).get_fdata()


# Generate the color map:
labels = np.unique(seg) 
label_map = {label: i for i, label in enumerate(labels)}
seg_mapped = np.vectorize(label_map.get)(seg)

# Find the midplane
x_mid,y_mid,z_mid = image.shape[0] // 2 -10, image.shape[1] // 2, image.shape[2] // 2

# plot the image
fig, axes = plt.subplots(ncols=3, figsize=(12, 12))


def overlay(ax, img_slice, seg_slice):

    # Show MRI scan
    ax.imshow(img_slice, cmap="gray")
    #  mask background so it doesn't wash out MRI
    seg_masked = np.ma.masked_where(seg_slice == 0, seg_slice)
    # plot masked segmentation
    ax.imshow(seg_masked,cmap="tab20",alpha=0.5)
    ax.axis("off")


overlay(axes[0], np.rot90(image[x_mid, :, :]), np.rot90(seg_mapped[x_mid, :, :]))
overlay(axes[1], np.rot90(image[:, y_mid, :]), np.rot90(seg_mapped[:, y_mid, :]))
overlay(axes[2], np.rot90(image[:, :, z_mid]), np.rot90(seg_mapped[:, :, z_mid]))
